In [1]:
nodes = [
    'Aw_in1',
    'Aw_in2',
    'B_x_xi',
    'Aw_out',
    'B_ix',
    'B_viii',
    'B_vii',
    'B_vi',
    'B_v',
    'B_iv'
]

node_to_idx = {
    node: i
    for i, node in enumerate(nodes)
}

In [2]:
node_to_idx

{'Aw_in1': 0,
 'Aw_in2': 1,
 'B_x_xi': 2,
 'Aw_out': 3,
 'B_ix': 4,
 'B_viii': 5,
 'B_vii': 6,
 'B_vi': 7,
 'B_v': 8,
 'B_iv': 9}

In [3]:
import random


def evaluate_shield_policy_worst_case_expected(
    solution,
    node_to_idx,
    coverage,
    attack_duration,
    target_values,
    adj_matrix,
    attacker_history_length=1,
    delays=(1,2,3,4),
    rollouts_num=10000,
    protect_current_vertex=False,
):

    num_nodes = len(target_values)

    # =========================
    # HELPERS
    # =========================

    def sample_from_distribution(dist):
        items = []
        probs = []

        for x, p in dist.items():
            if p > 1e-12:
                items.append(x)
                probs.append(p)

        return random.choices(items, weights=probs)[0]

    def current_vertex(state):
        return state[0][-1][0]


    def expected_capture_probability(path, target, tau):
        survive_prob = 1.0
        
        if protect_current_vertex:  
            survive_prob *= (
                1.0 - coverage[path[0]][target]
            )
        for i in range(len(path) - 1):

            pos = path[i]
            next_pos = path[i + 1]

            tau -= adj_matrix[pos][next_pos]
            if tau >= 0:
                survive_prob *= 1.0 - coverage[next_pos][target]
            else:
                break
        return 1.0 - survive_prob

    # =========================
    # HORIZON
    # =========================

    max_tau = max(attack_duration)
    max_delay = max(delays)

    horizon = int(max_tau + max_delay + 1)

    # =========================
    # MAIN LOOP
    # =========================

    observation_rewards = {}

    for _ in range(rollouts_num):

        state = sample_from_distribution(solution.stationary)
        state_rollout = [state]

        for _ in range(horizon):

            transition = solution.transition[state]
            state = sample_from_distribution(transition)

            state_rollout.append(state)

        vertex_rollout = [node_to_idx[current_vertex(s)]for s in state_rollout]
        
        
        # ---------------------
        # evaluate attacks
        # ---------------------

        for delay in delays:

            if attacker_history_length == -1:
                observation = tuple(vertex_rollout[:delay])
            else:
                observation = tuple(vertex_rollout[max(0, delay - attacker_history_length):delay])


            for target in range(num_nodes):

                if target_values[target] == 0:
                    continue

                tau = attack_duration[target]

                attack_path = vertex_rollout[(delay - 1):]

                capture_prob = expected_capture_probability(attack_path,target,tau)

                reward = capture_prob * target_values[target]

                key = (observation,target)

                if key not in observation_rewards:
                    observation_rewards[key] = []

                observation_rewards[key].append(reward)



    # =========================
    # WORST CASE
    # =========================

    obs_target_rewards = {}
    obs_target_counts = {}

    # key = (observation, target)
    for key, vals in observation_rewards.items():

        mean_reward = (sum(vals) / len(vals))

        obs_target_rewards[key] = mean_reward
        obs_target_counts[key] = len(vals)

    worst_pair = None
    worst_value = float("inf")
    count = 0

    for pair, value in obs_target_rewards.items():

        if value < worst_value:
            worst_value = value
            worst_pair = pair
            count = obs_target_counts[pair]

    print("\nWorst case:", worst_pair)
    print("Worst-case value:", worst_value)
    print(f"\nWorst-case value in %: {worst_value * 100:.2f}") 
    print("\nCount:", count)
    print(obs_target_rewards)
    print(obs_target_counts)
    return worst_value

# Gdynia version 1

protect_current_vertex = false

In [6]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "gdynia_1000_1_1_3.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [0, 1, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 1, 0, 1, 0, 0, 0, 0, 1],
      [0, 0, 0, 1, 0, 0, 0, 0, 1, 1],
      [0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 1, 0, 1, 0],
      [0, 0, 0, 0, 1, 0, 0, 1, 0, 1],
      [0, 0, 0, 1, 1, 0, 0, 0, 1, 0]
    ]
coverage_matrix = [
      [1, 0.5, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 1, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0.5],
      [0, 0, 0, 0.5, 1, 0, 0, 0, 0.5, 0.5],
      [0, 0, 0, 0, 0, 1, 0.5, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 1, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 0.5, 1, 0.5, 0],
      [0, 0, 0, 0, 0.5, 0, 0, 0.5, 1, 0.5],
      [0, 0, 0, 0.5, 0.5, 0, 0, 0, 0.5, 1]
    ]
targets = [0, 0, 1, 0, 1, 1, 1, 1, 1, 1]
attack_duration = [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]


evaluate_shield_policy_worst_case_expected(solution, node_to_idx, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, delays=(1,2,3,4), rollouts_num=1000000, protect_current_vertex=False)



Worst case: ((7,), 5)
Worst-case value: 0.0

Worst-case value in %: 0.00

Count: 570965
{((9,), 2): 0.25030979584978574, ((9,), 4): 0.843957405696043, ((9,), 5): 0.24969020415021423, ((9,), 6): 0.24969020415021423, ((9,), 7): 0.7496902041502143, ((9,), 8): 1.0, ((9,), 9): 0.8437749411912963, ((3,), 2): 0.12539008691024953, ((3,), 4): 0.9377584469134476, ((3,), 5): 0.2497447478344524, ((3,), 6): 0.2497447478344524, ((3,), 7): 0.6243546609242029, ((3,), 8): 0.9373049565448752, ((3,), 9): 0.9373691791693263, ((8,), 2): 0.2502176315729552, ((8,), 4): 0.9374056345996542, ((8,), 5): 0.24978236842704482, ((8,), 6): 0.24978236842704482, ((8,), 7): 0.49956473685408964, ((8,), 8): 0.8748911842135224, ((8,), 9): 0.8959991478651087, ((7,), 2): 0.5, ((7,), 4): 0.9583363253439353, ((7,), 5): 0.0, ((7,), 6): 0.0, ((7,), 7): 0.5, ((7,), 8): 1.0, ((7,), 9): 0.9166636746560647, ((4,), 2): 0.31256115283421254, ((4,), 4): 0.8437141772271436, ((4,), 5): 0.2500052463557501, ((4,), 6): 0.2500052463557501, (

0.0

# Gdynia version 1

protect_current_vertex = True

In [7]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "gdynia_1000_1_1_3.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [0, 1, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 1, 0, 1, 0, 0, 0, 0, 1],
      [0, 0, 0, 1, 0, 0, 0, 0, 1, 1],
      [0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 1, 0, 1, 0],
      [0, 0, 0, 0, 1, 0, 0, 1, 0, 1],
      [0, 0, 0, 1, 1, 0, 0, 0, 1, 0]
    ]
coverage_matrix = [
      [1, 0.5, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 1, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0.5],
      [0, 0, 0, 0.5, 1, 0, 0, 0, 0.5, 0.5],
      [0, 0, 0, 0, 0, 1, 0.5, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 1, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 0.5, 1, 0.5, 0],
      [0, 0, 0, 0, 0.5, 0, 0, 0.5, 1, 0.5],
      [0, 0, 0, 0.5, 0.5, 0, 0, 0, 0.5, 1]
    ]
targets = [0, 0, 1, 0, 1, 1, 1, 1, 1, 1]
attack_duration = [3, 3, 3, 3, 3, 3, 3, 3, 3, 3]


evaluate_shield_policy_worst_case_expected(solution, node_to_idx, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, delays=(1,2,3,4), rollouts_num=1000000, protect_current_vertex=True)



Worst case: ((9,), 2)
Worst-case value: 0.24953462654025824

Worst-case value in %: 24.95

Count: 762291
{((7,), 2): 0.5, ((7,), 4): 0.9584603958749459, ((7,), 5): 0.5, ((7,), 6): 0.5, ((7,), 7): 1.0, ((7,), 8): 1.0, ((7,), 9): 0.9165396041250541, ((8,), 2): 0.24992079509472206, ((8,), 4): 0.9687849529934203, ((8,), 5): 0.2500792049052779, ((8,), 6): 0.2500792049052779, ((8,), 7): 0.7500792049052779, ((8,), 8): 1.0, ((8,), 9): 0.9478873294906556, ((4,), 2): 0.31220947018911505, ((4,), 4): 1.0, ((4,), 5): 0.25021391918316993, ((4,), 6): 0.25021391918316993, ((4,), 7): 0.5004278383663399, ((4,), 8): 0.90634178510965, ((4,), 9): 0.9687346614696116, ((3,), 2): 0.5624297800738163, ((3,), 4): 0.9687924928992561, ((3,), 5): 0.249805746746258, ((3,), 6): 0.249805746746258, ((3,), 7): 0.6249461865986256, ((3,), 8): 0.9375702199261837, ((3,), 9): 0.9687560704141794, ((9,), 2): 0.24953462654025824, ((9,), 4): 0.9218070920422778, ((9,), 5): 0.25046537345974174, ((9,), 6): 0.25046537345974174, ((9

0.24953462654025824

# Gdynia version 2

protect_current_vertex = false

In [8]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "gdynia_1000_1_1_4.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [0, 1, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 1, 0, 1, 0, 0, 0, 0, 1],
      [0, 0, 0, 1, 0, 0, 0, 0, 1, 1],
      [0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 1, 0, 1, 0],
      [0, 0, 0, 0, 1, 0, 0, 1, 0, 1],
      [0, 0, 0, 1, 1, 0, 0, 0, 1, 0]
    ]
coverage_matrix = [
      [1, 0.5, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 1, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0.5],
      [0, 0, 0, 0.5, 1, 0, 0, 0, 0.5, 0.5],
      [0, 0, 0, 0, 0, 1, 0.5, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 1, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 0.5, 1, 0.5, 0],
      [0, 0, 0, 0, 0.5, 0, 0, 0.5, 1, 0.5],
      [0, 0, 0, 0.5, 0.5, 0, 0, 0, 0.5, 1]
    ]
targets = [0, 0, 1, 0, 1, 1, 1, 1, 1, 1]
attack_duration = [4, 4, 4, 4, 4, 4, 4, 4, 4, 4]


evaluate_shield_policy_worst_case_expected(solution, node_to_idx, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, delays=(1,2,3,4), rollouts_num=1000000, protect_current_vertex=False)



Worst case: ((7,), 6)
Worst-case value: 0.4314409782936674

Worst-case value in %: 43.14

Count: 792073
{((3,), 2): 0.44630134954964257, ((3,), 4): 0.833249756053643, ((3,), 5): 0.4609066860499756, ((3,), 6): 0.4608375072323309, ((3,), 7): 0.7471775704399243, ((3,), 8): 0.9923823860800303, ((3,), 9): 0.9844872293916633, ((2,), 2): 0.5, ((2,), 4): 0.8924009221403205, ((2,), 5): 0.4724387477990987, ((2,), 6): 0.4724387477990987, ((2,), 7): 0.9515279477155391, ((2,), 8): 0.9947722999791101, ((2,), 9): 0.9894893909099048, ((9,), 2): 0.46067552906639586, ((9,), 4): 0.6981725462612272, ((9,), 5): 0.4746080039067811, ((9,), 6): 0.47457644166604235, ((9,), 7): 0.5223758944947128, ((9,), 8): 0.77484668034595, ((9,), 9): 0.7513437363866159, ((8,), 2): 0.46158939500789514, ((8,), 4): 0.6923396322319492, ((8,), 5): 0.46043850451002616, ((8,), 6): 0.46041635129097885, ((8,), 7): 0.5483795535634068, ((8,), 8): 0.7892448890677556, ((8,), 9): 0.7384500961818927, ((7,), 2): 0.4616859809638758, ((7,), 

0.4314409782936674

# Gdynia version 2

protect_current_vertex = True

In [9]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "gdynia_1000_1_1_4.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [0, 1, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 0, 1, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 0, 1, 0, 0, 0, 0, 0, 0],
      [1, 1, 1, 0, 1, 0, 0, 0, 0, 1],
      [0, 0, 0, 1, 0, 0, 0, 0, 1, 1],
      [0, 0, 0, 0, 0, 0, 1, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 1, 1, 0, 1, 0],
      [0, 0, 0, 0, 1, 0, 0, 1, 0, 1],
      [0, 0, 0, 1, 1, 0, 0, 0, 1, 0]
    ]
coverage_matrix = [
      [1, 0.5, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 1, 0.5, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0, 0],
      [0.5, 0.5, 0.5, 1, 0.5, 0, 0, 0, 0, 0.5],
      [0, 0, 0, 0.5, 1, 0, 0, 0, 0.5, 0.5],
      [0, 0, 0, 0, 0, 1, 0.5, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 1, 0.5, 0, 0],
      [0, 0, 0, 0, 0, 0.5, 0.5, 1, 0.5, 0],
      [0, 0, 0, 0, 0.5, 0, 0, 0.5, 1, 0.5],
      [0, 0, 0, 0.5, 0.5, 0, 0, 0, 0.5, 1]
    ]
targets = [0, 0, 1, 0, 1, 1, 1, 1, 1, 1]
attack_duration = [4, 4, 4, 4, 4, 4, 4, 4, 4, 4]


evaluate_shield_policy_worst_case_expected(solution, node_to_idx, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, delays=(1,2,3,4), rollouts_num=1000000, protect_current_vertex=True)



Worst case: ((3,), 6)
Worst-case value: 0.4605076257712811

Worst-case value in %: 46.05

Count: 755399
{((9,), 2): 0.4608586769521061, ((9,), 4): 0.849140837449359, ((9,), 5): 0.474318948054831, ((9,), 6): 0.4744044480825795, ((9,), 7): 0.5220780842444087, ((9,), 8): 0.887391650826905, ((9,), 9): 1.0, ((8,), 2): 0.46078519516571814, ((8,), 4): 0.8460878488972857, ((8,), 5): 0.4610332089005371, ((8,), 6): 0.4611215763020182, ((8,), 7): 0.7745519741954872, ((8,), 8): 1.0, ((8,), 9): 0.8691068642073229, ((7,), 2): 0.46097858740972986, ((7,), 4): 0.8491398010841243, ((7,), 5): 0.716036772699641, ((7,), 6): 0.7161433437844464, ((7,), 7): 1.0, ((7,), 8): 1.0, ((7,), 9): 0.9464731749859376, ((3,), 2): 0.7231830463106252, ((3,), 4): 0.9166145970540073, ((3,), 5): 0.4605754707115048, ((3,), 6): 0.4605076257712811, ((3,), 7): 0.7468324686688756, ((3,), 8): 0.9923080385332784, ((3,), 9): 0.9923269854739019, ((2,), 2): 1.0, ((2,), 4): 0.8924105637564267, ((2,), 5): 0.4720941613179481, ((2,), 6):

0.4605076257712811

# San Francisco


In [1]:
import random

def evaluate_shield_policy_worst_case_expected_sf(
    solution,
    coverage,
    attack_duration,
    target_values,
    adj_matrix,
    attacker_history_length=1,
    delays=(1, 2, 3),
    rollouts_num=10000,
    protect_current_vertex=False,
):

    num_nodes = len(target_values)

    # =========================
    # HELPERS
    # =========================

    def sample_from_distribution(dist):
        items = []
        probs = []

        for x, p in dist.items():
            if p > 1e-12:
                items.append(x)
                probs.append(p)

        return random.choices(items, weights=probs)[0]

    def current_vertex(state):
        return state[0][-1][0]

    def expected_capture_probability(path, target, tau):
        survive_prob = 1.0

        if protect_current_vertex:
            survive_prob *= (1.0 - coverage[path[0]][target])

        for i in range(len(path) - 1):

            pos = path[i]
            next_pos = path[i + 1]

            tau -= adj_matrix[pos][next_pos]

            if tau >= 0:
                survive_prob *= (1.0 - coverage[next_pos][target])
            else:
                break

        return 1.0 - survive_prob

    # =========================
    # HORIZON
    # =========================

    max_tau = max(attack_duration)
    max_delay = max(delays)
    horizon = int(max_tau + max_delay + 1)

    # =========================
    # MAIN LOOP
    # =========================

    observation_rewards = {}

    for _ in range(rollouts_num):
        if _ % 10000 == 0:
            print(f"Rollout {_}/{rollouts_num}")
        state = sample_from_distribution(solution.stationary)

        # Collect only real graph vertices
        vertex_rollout = []

        while len(vertex_rollout) < horizon:

            v = current_vertex(state)
            # Ignore intermediate edge states
            if v >= 0:
                vertex_rollout.append(v)

            transition = solution.transition[state]
            state = sample_from_distribution(transition)

        # ---------------------
        # evaluate attacks
        # ---------------------
        for delay in delays:

            if attacker_history_length == -1:
                observation = tuple(vertex_rollout[:delay])
            else:
                observation = tuple(
                    vertex_rollout[max(0, delay - attacker_history_length):delay]
                )

            for target in range(num_nodes):

                if target_values[target] == 0:
                    continue

                tau = attack_duration[target]

                attack_path = vertex_rollout[(delay - 1):]

                capture_prob = expected_capture_probability(
                    attack_path,
                    target,
                    tau
                )

                reward = capture_prob * target_values[target]

                key = (observation, target)

                if key not in observation_rewards:
                    observation_rewards[key] = []

                observation_rewards[key].append(reward)

    # =========================
    # WORST CASE
    # =========================

    obs_target_rewards = {}
    obs_target_counts = {}

    for key, vals in observation_rewards.items():

        mean_reward = sum(vals) / len(vals)

        obs_target_rewards[key] = mean_reward
        obs_target_counts[key] = len(vals)

    worst_pair = None
    worst_value = float("inf")
    count = 0

    for pair, value in obs_target_rewards.items():

        if value < worst_value:
            worst_value = value
            worst_pair = pair
            count = obs_target_counts[pair]
    print("\n\n\n\n")
    print("\nWorst case:", worst_pair)
    print("Worst-case value:", worst_value)
    print(f"\nWorst-case value in %: {worst_value * 100:.2f}")
    print("\nCount:", count)
    print(obs_target_rewards)
    print(obs_target_counts)
    return worst_value

# Delay 1
protect_current_vertex=False

In [2]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "san_francisco.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [1, 3, 3, 5, 4, 6, 3, 5, 7, 4, 6, 6],
      [3, 1, 5, 4, 2, 4, 4, 5, 5, 3, 5, 5],
      [3, 5, 1, 7, 6, 8, 3, 4, 9, 4, 8, 7],
      [6, 4, 7, 1, 5, 6, 4, 7, 5, 6, 6, 7],
      [4, 3, 6, 5, 1, 3, 5, 5, 6, 3, 4, 4],
      [6, 4, 8, 5, 3, 1, 6, 7, 3, 6, 2, 3],
      [2, 5, 3, 5, 6, 7, 1, 5, 7, 5, 7, 8],
      [3, 5, 2, 7, 6, 7, 3, 1, 9, 3, 7, 5],
      [8, 6, 9, 4, 6, 4, 6, 9, 1, 8, 5, 7],
      [4, 3, 4, 6, 3, 5, 5, 3, 7, 1, 5, 3],
      [6, 4, 8, 6, 4, 2, 6, 6, 4, 5, 1, 3],
      [6, 4, 6, 6, 3, 3, 6, 4, 5, 3, 2, 1]
    ]
coverage_matrix = [
      [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    ]
targets = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
attack_duration = [8, 6, 11, 10, 6, 10, 9, 10, 11, 9, 10, 8]


evaluate_shield_policy_worst_case_expected_sf(solution, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, delays=(1,), rollouts_num=100000, protect_current_vertex=False)


Rollout 0/100000
Rollout 10000/100000
Rollout 20000/100000
Rollout 30000/100000
Rollout 40000/100000
Rollout 50000/100000
Rollout 60000/100000
Rollout 70000/100000
Rollout 80000/100000
Rollout 90000/100000






Worst case: ((4,), 4)
Worst-case value: 0.0

Worst-case value in %: 0.00

Count: 10782
{((4,), 0): 0.19040994249675386, ((4,), 1): 0.221480244852532, ((4,), 2): 0.16703765535151177, ((4,), 3): 0.19337785197551474, ((4,), 4): 0.0, ((4,), 5): 0.18224819143016138, ((4,), 6): 0.22046002596920794, ((4,), 7): 0.2062697087738824, ((4,), 8): 0.11352253756260434, ((4,), 9): 0.28890743832313115, ((4,), 10): 0.20385828232238917, ((4,), 11): 0.15906139862734187, ((1,), 0): 0.18017775383786697, ((1,), 1): 0.009605889218062663, ((1,), 2): 0.1793697818475626, ((1,), 3): 0.23951880779244097, ((1,), 4): 0.16958434329832123, ((1,), 5): 0.1782027111949008, ((1,), 6): 0.20262142023520963, ((1,), 7): 0.1944519256665769, ((1,), 8): 0.1466020289074423, ((1,), 9): 0.19552922165364933, ((1,), 10): 0.19

0.0

# Delay 1
protect_current_vertex=True

In [3]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "san_francisco.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [1, 3, 3, 5, 4, 6, 3, 5, 7, 4, 6, 6],
      [3, 1, 5, 4, 2, 4, 4, 5, 5, 3, 5, 5],
      [3, 5, 1, 7, 6, 8, 3, 4, 9, 4, 8, 7],
      [6, 4, 7, 1, 5, 6, 4, 7, 5, 6, 6, 7],
      [4, 3, 6, 5, 1, 3, 5, 5, 6, 3, 4, 4],
      [6, 4, 8, 5, 3, 1, 6, 7, 3, 6, 2, 3],
      [2, 5, 3, 5, 6, 7, 1, 5, 7, 5, 7, 8],
      [3, 5, 2, 7, 6, 7, 3, 1, 9, 3, 7, 5],
      [8, 6, 9, 4, 6, 4, 6, 9, 1, 8, 5, 7],
      [4, 3, 4, 6, 3, 5, 5, 3, 7, 1, 5, 3],
      [6, 4, 8, 6, 4, 2, 6, 6, 4, 5, 1, 3],
      [6, 4, 6, 6, 3, 3, 6, 4, 5, 3, 2, 1]
    ]
coverage_matrix = [
      [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    ]
targets = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
attack_duration = [8, 6, 11, 10, 6, 10, 9, 10, 11, 9, 10, 8]


evaluate_shield_policy_worst_case_expected_sf(solution, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, delays=(1,), rollouts_num=100000, protect_current_vertex=True)


Rollout 0/100000
Rollout 10000/100000
Rollout 20000/100000
Rollout 30000/100000
Rollout 40000/100000
Rollout 50000/100000
Rollout 60000/100000
Rollout 70000/100000
Rollout 80000/100000
Rollout 90000/100000






Worst case: ((4,), 8)
Worst-case value: 0.11540600667408231

Worst-case value in %: 11.54

Count: 10788
{((9,), 0): 0.218953794921604, ((9,), 1): 0.1813514638545858, ((9,), 2): 0.213819897322048, ((9,), 3): 0.18870542528097684, ((9,), 4): 0.1752462883307895, ((9,), 5): 0.21076730956014986, ((9,), 6): 0.18190647981129457, ((9,), 7): 0.21395865131122518, ((9,), 8): 0.17191619259053698, ((9,), 9): 1.0, ((9,), 10): 0.18565283751907868, ((9,), 11): 0.2503121964756487, ((6,), 0): 0.6248, ((6,), 1): 0.21074285714285715, ((6,), 2): 0.19257142857142856, ((6,), 3): 0.13497142857142858, ((6,), 4): 0.20948571428571428, ((6,), 5): 0.19988571428571428, ((6,), 6): 1.0, ((6,), 7): 0.15428571428571428, ((6,), 8): 0.13885714285714285, ((6,), 9): 0.21394285714285713, ((6,), 10): 0.2408, ((6,), 11

0.11540600667408231

# Delay 2
protect_current_vertex=False

In [4]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "san_francisco.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [1, 3, 3, 5, 4, 6, 3, 5, 7, 4, 6, 6],
      [3, 1, 5, 4, 2, 4, 4, 5, 5, 3, 5, 5],
      [3, 5, 1, 7, 6, 8, 3, 4, 9, 4, 8, 7],
      [6, 4, 7, 1, 5, 6, 4, 7, 5, 6, 6, 7],
      [4, 3, 6, 5, 1, 3, 5, 5, 6, 3, 4, 4],
      [6, 4, 8, 5, 3, 1, 6, 7, 3, 6, 2, 3],
      [2, 5, 3, 5, 6, 7, 1, 5, 7, 5, 7, 8],
      [3, 5, 2, 7, 6, 7, 3, 1, 9, 3, 7, 5],
      [8, 6, 9, 4, 6, 4, 6, 9, 1, 8, 5, 7],
      [4, 3, 4, 6, 3, 5, 5, 3, 7, 1, 5, 3],
      [6, 4, 8, 6, 4, 2, 6, 6, 4, 5, 1, 3],
      [6, 4, 6, 6, 3, 3, 6, 4, 5, 3, 2, 1]
    ]
coverage_matrix = [
      [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    ]
targets = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
attack_duration = [8, 6, 11, 10, 6, 10, 9, 10, 11, 9, 10, 8]


evaluate_shield_policy_worst_case_expected_sf(solution, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, delays=(1,2), rollouts_num=100000, protect_current_vertex=False)


Rollout 0/100000
Rollout 10000/100000
Rollout 20000/100000
Rollout 30000/100000
Rollout 40000/100000
Rollout 50000/100000
Rollout 60000/100000
Rollout 70000/100000
Rollout 80000/100000
Rollout 90000/100000






Worst case: ((7,), 7)
Worst-case value: 0.0

Worst-case value in %: 0.00

Count: 16956
{((7,), 0): 0.1767515923566879, ((7,), 1): 0.1956829440905874, ((7,), 2): 0.36924982307147913, ((7,), 3): 0.14578910120311395, ((7,), 4): 0.1968624675631045, ((7,), 5): 0.22581976881339938, ((7,), 6): 0.14248643548006606, ((7,), 7): 0.0, ((7,), 8): 0.19361877801368246, ((7,), 9): 0.2357867421561689, ((7,), 10): 0.17834394904458598, ((7,), 11): 0.19680349138947864, ((11,), 0): 0.22247491638795985, ((11,), 1): 0.20098104793756968, ((11,), 2): 0.1485395763656633, ((11,), 3): 0.21525083612040133, ((11,), 4): 0.19117056856187292, ((11,), 5): 0.21177257525083612, ((11,), 6): 0.165752508361204, ((11,), 7): 0.13391304347826086, ((11,), 8): 0.19585284280936455, ((11,), 9): 0.19393534002229654, ((11,),

0.0

# Delay 2
protect_current_vertex=True

In [5]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "san_francisco.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [1, 3, 3, 5, 4, 6, 3, 5, 7, 4, 6, 6],
      [3, 1, 5, 4, 2, 4, 4, 5, 5, 3, 5, 5],
      [3, 5, 1, 7, 6, 8, 3, 4, 9, 4, 8, 7],
      [6, 4, 7, 1, 5, 6, 4, 7, 5, 6, 6, 7],
      [4, 3, 6, 5, 1, 3, 5, 5, 6, 3, 4, 4],
      [6, 4, 8, 5, 3, 1, 6, 7, 3, 6, 2, 3],
      [2, 5, 3, 5, 6, 7, 1, 5, 7, 5, 7, 8],
      [3, 5, 2, 7, 6, 7, 3, 1, 9, 3, 7, 5],
      [8, 6, 9, 4, 6, 4, 6, 9, 1, 8, 5, 7],
      [4, 3, 4, 6, 3, 5, 5, 3, 7, 1, 5, 3],
      [6, 4, 8, 6, 4, 2, 6, 6, 4, 5, 1, 3],
      [6, 4, 6, 6, 3, 3, 6, 4, 5, 3, 2, 1]
    ]
coverage_matrix = [
      [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    ]
targets = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
attack_duration = [8, 6, 11, 10, 6, 10, 9, 10, 11, 9, 10, 8]


evaluate_shield_policy_worst_case_expected_sf(solution, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, delays=(1,2), rollouts_num=100000, protect_current_vertex=True)


Rollout 0/100000
Rollout 10000/100000
Rollout 20000/100000
Rollout 30000/100000
Rollout 40000/100000
Rollout 50000/100000
Rollout 60000/100000
Rollout 70000/100000
Rollout 80000/100000
Rollout 90000/100000






Worst case: ((11,), 7)
Worst-case value: 0.13726973391474592

Worst-case value in %: 13.73

Count: 22474
{((9,), 0): 0.22237220494160456, ((9,), 1): 0.1961382710252949, ((9,), 2): 0.18704149304536652, ((9,), 3): 0.21468396032631024, ((9,), 4): 0.19983567110745937, ((9,), 5): 0.1923234931627443, ((9,), 6): 0.19654909325664652, ((9,), 7): 0.19742942660954282, ((9,), 8): 0.16708727037971713, ((9,), 9): 1.0, ((9,), 10): 0.18698280415517343, ((9,), 11): 0.2001878044486179, ((4,), 0): 0.20962903709629038, ((4,), 1): 0.22472752724727527, ((4,), 2): 0.18748125187481252, ((4,), 3): 0.18318168183181682, ((4,), 4): 1.0, ((4,), 5): 0.17613238676132387, ((4,), 6): 0.22962703729627038, ((4,), 7): 0.18588141185881413, ((4,), 8): 0.1498850114988501, ((4,), 9): 0.2753724627537246, ((4,), 10): 0

0.13726973391474592

# Delay 3
protect_current_vertex=False

In [6]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "san_francisco.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [1, 3, 3, 5, 4, 6, 3, 5, 7, 4, 6, 6],
      [3, 1, 5, 4, 2, 4, 4, 5, 5, 3, 5, 5],
      [3, 5, 1, 7, 6, 8, 3, 4, 9, 4, 8, 7],
      [6, 4, 7, 1, 5, 6, 4, 7, 5, 6, 6, 7],
      [4, 3, 6, 5, 1, 3, 5, 5, 6, 3, 4, 4],
      [6, 4, 8, 5, 3, 1, 6, 7, 3, 6, 2, 3],
      [2, 5, 3, 5, 6, 7, 1, 5, 7, 5, 7, 8],
      [3, 5, 2, 7, 6, 7, 3, 1, 9, 3, 7, 5],
      [8, 6, 9, 4, 6, 4, 6, 9, 1, 8, 5, 7],
      [4, 3, 4, 6, 3, 5, 5, 3, 7, 1, 5, 3],
      [6, 4, 8, 6, 4, 2, 6, 6, 4, 5, 1, 3],
      [6, 4, 6, 6, 3, 3, 6, 4, 5, 3, 2, 1]
    ]
coverage_matrix = [
      [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    ]
targets = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
attack_duration = [8, 6, 11, 10, 6, 10, 9, 10, 11, 9, 10, 8]


evaluate_shield_policy_worst_case_expected_sf(solution, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, delays=(1,2,3), rollouts_num=100000, protect_current_vertex=False)


Rollout 0/100000
Rollout 10000/100000
Rollout 20000/100000
Rollout 30000/100000
Rollout 40000/100000
Rollout 50000/100000
Rollout 60000/100000
Rollout 70000/100000
Rollout 80000/100000
Rollout 90000/100000






Worst case: ((6,), 6)
Worst-case value: 0.0

Worst-case value in %: 0.00

Count: 22574
{((6,), 0): 0.5757951625764154, ((6,), 1): 0.22034198635598476, ((6,), 2): 0.17692921059626118, ((6,), 3): 0.16000708780012404, ((6,), 4): 0.2010720297687605, ((6,), 5): 0.20665367236643928, ((6,), 6): 0.0, ((6,), 7): 0.17041729423230265, ((6,), 8): 0.1627536103481882, ((6,), 9): 0.1908390183396828, ((6,), 10): 0.22570213519978738, ((6,), 11): 0.21502613626295738, ((0,), 0): 0.0, ((0,), 1): 0.2641579044456742, ((0,), 2): 0.19155137428518723, ((0,), 3): 0.20822726434237226, ((0,), 4): 0.18985427042980998, ((0,), 5): 0.19055524810920493, ((0,), 6): 0.2143884892086331, ((0,), 7): 0.19704851503412654, ((0,), 8): 0.17719977863862757, ((0,), 9): 0.2030252720900203, ((0,), 10): 0.18454159749123777, 

0.0

# Delay 3
protect_current_vertex=True

In [7]:
import pickle
import stackelberg_games.patrolling as sgp

file = sgp.directories.strategy_dir + "san_francisco.pickle"

with open(file, "rb") as f:
    solution = pickle.load(f)

adj_matrix = [
      [1, 3, 3, 5, 4, 6, 3, 5, 7, 4, 6, 6],
      [3, 1, 5, 4, 2, 4, 4, 5, 5, 3, 5, 5],
      [3, 5, 1, 7, 6, 8, 3, 4, 9, 4, 8, 7],
      [6, 4, 7, 1, 5, 6, 4, 7, 5, 6, 6, 7],
      [4, 3, 6, 5, 1, 3, 5, 5, 6, 3, 4, 4],
      [6, 4, 8, 5, 3, 1, 6, 7, 3, 6, 2, 3],
      [2, 5, 3, 5, 6, 7, 1, 5, 7, 5, 7, 8],
      [3, 5, 2, 7, 6, 7, 3, 1, 9, 3, 7, 5],
      [8, 6, 9, 4, 6, 4, 6, 9, 1, 8, 5, 7],
      [4, 3, 4, 6, 3, 5, 5, 3, 7, 1, 5, 3],
      [6, 4, 8, 6, 4, 2, 6, 6, 4, 5, 1, 3],
      [6, 4, 6, 6, 3, 3, 6, 4, 5, 3, 2, 1]
    ]
coverage_matrix = [
      [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0],
      [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
    ]
targets = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
attack_duration = [8, 6, 11, 10, 6, 10, 9, 10, 11, 9, 10, 8]


evaluate_shield_policy_worst_case_expected_sf(solution, coverage_matrix, attack_duration, targets, adj_matrix, attacker_history_length=1, delays=(1,2,3), rollouts_num=100000, protect_current_vertex=True)


Rollout 0/100000
Rollout 10000/100000
Rollout 20000/100000
Rollout 30000/100000
Rollout 40000/100000
Rollout 50000/100000
Rollout 60000/100000
Rollout 70000/100000
Rollout 80000/100000
Rollout 90000/100000






Worst case: ((7,), 3)
Worst-case value: 0.14270377405235776

Worst-case value in %: 14.27

Count: 24218
{((8,), 0): 0.149655839965584, ((8,), 1): 0.19541836954183694, ((8,), 2): 0.1699290169929017, ((8,), 3): 0.19563346956334696, ((8,), 4): 0.21606797160679717, ((8,), 5): 0.21305657130565714, ((8,), 6): 0.149655839965584, ((8,), 7): 0.19450419445041944, ((8,), 8): 1.0, ((8,), 9): 0.21606797160679717, ((8,), 10): 0.20837814583781458, ((8,), 11): 0.21219617121961712, ((6,), 0): 0.5752095860948954, ((6,), 1): 0.2183206776982838, ((6,), 2): 0.17881753939340736, ((6,), 3): 0.15968046350348944, ((6,), 4): 0.1980424000351139, ((6,), 5): 0.20554799631304044, ((6,), 6): 1.0, ((6,), 7): 0.16907343194487118, ((6,), 8): 0.16213843655357066, ((6,), 9): 0.19317034631084581, ((6,), 10): 0.226

0.14270377405235776